# Trade Surveillance — Exploratory Data Analysis

This notebook walks the simulated trade blotter that powers the dashboard: what the book
looks like, how the five surveillance rules behave on it, and what an analyst would take
away from a first pass.

Everything is **synthetic**. Trader and counterparty names come from Faker; there is no
real client, counterparty or employee data anywhere in this project.

The notebook imports the same `src` modules the Streamlit app uses, so any number here can
be reproduced on the dashboard and vice versa.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio

# Run from anywhere inside the repo.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import (
    ENTITY_LEVEL_FLAGS,
    FLAG_COLUMNS,
    FLAG_LABELS,
    PRODUCT_SETTLEMENT_WINDOW,
    SurveillanceConfig,
    TRADES_CSV,
    TRADE_LEVEL_FLAGS,
)
from src.data_cleaning import clean_trades, load_trades
from src.sqlite_views import desk_exception_league_table, monthly_exception_trend
from src.surveillance_rules import (
    exception_summary,
    flag_counterparty_concentration,
    flag_high_trade_volume,
    flag_large_notional,
    flag_late_amendment_cancellation,
    flag_settlement_risk,
    run_all_rules,
)
from src.visualizations import (
    exposure_by_counterparty_chart,
    settlement_timing_chart,
    trade_volume_trend_chart,
    trader_desk_activity_chart,
)

# Load plotly.js from the CDN: keeps the committed notebook small (a full
# embed adds ~4MB) while still rendering in any Jupyter session with a network.
pio.renderers.default = "notebook_connected"
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 40)
print(f"pandas {pd.__version__} | numpy {np.__version__}")

pandas 2.2.3 | numpy 1.26.4


## 1. Load and clean

`load_trades` enforces the schema and dtypes and raises on anything unparseable;
`clean_trades` then de-duplicates, normalises the categoricals and derives the three
helper columns the rules depend on (`trade_month`, `days_to_settlement`,
`days_to_amendment`). The cleaning report tells us what, if anything, had to be repaired.

In [2]:
raw = load_trades(TRADES_CSV)
df = clean_trades(raw)

report = df.attrs["cleaning_report"]
for line in report.summary_lines():
    print(line)
print(f"\nnothing needed repairing: {report.is_clean()}")

df.head()

3,600 rows in -> 3,600 rows out

nothing needed repairing: True


,trade_id,trade_date,trader,desk,product_type,counterparty,buy_sell,quantity,price,notional_value,trade_status,amendment_flag,cancellation_flag,booking_datetime,amendment_datetime,settlement_date,trade_month,days_to_settlement,days_to_amendment,is_lifecycle_event
0,T00001,2026-01-01,Heather Garrett,Commodities,Swap,Crawford-Garrett,Buy,8,"437,897.82","3,503,182.56",Booked,False,False,2026-01-01 07:22:00,NaT,2026-01-05,2026-01,2.00,NaN,False
1,T00002,2026-01-01,Heather Garrett,Commodities,Option,Chang-Peterson,Sell,3284,373.67,"1,227,132.28",Booked,False,False,2026-01-01 10:08:00,NaT,2026-01-02,2026-01,1.00,NaN,False
2,T00003,2026-01-01,Kevin Gutierrez,Equities,Option,Ellis Group,Buy,590,741.71,"437,608.90",Cancelled,True,True,2026-01-01 10:13:00,2026-01-01 18:34:00,2026-01-02,2026-01,1.00,0.00,True
3,T00004,2026-01-01,Pamela Hahn,FX,FX Forward,Mclaughlin PLC,Sell,2136554,0.92,"1,970,330.10",Amended,True,False,2026-01-01 11:53:00,2026-01-01 17:39:00,2026-01-30,2026-01,21.00,0.00,True
4,T00005,2026-01-01,Courtney Wagner,Equities,Equity,Chang-Peterson,Buy,536,403.70,"216,383.20",Booked,False,False,2026-01-01 12:10:00,NaT,2026-01-02,2026-01,1.00,NaN,False


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3600 entries, 0 to 3599
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   trade_id            3600 non-null   object        
 1   trade_date          3600 non-null   datetime64[ns]
 2   trader              3600 non-null   object        
 3   desk                3600 non-null   object        
 4   product_type        3600 non-null   object        
 5   counterparty        3600 non-null   object        
 6   buy_sell            3600 non-null   object        
 7   quantity            3600 non-null   int64         
 8   price               3600 non-null   float64       
 9   notional_value      3600 non-null   float64       
 10  trade_status        3600 non-null   object        
 11  amendment_flag      3600 non-null   bool          
 12  cancellation_flag   3600 non-null   bool          
 13  booking_datetime    3600 non-null   datetime64[n

## 2. What does the book look like?

The blotter covers six months of activity across five desks, five product types, fifteen
traders and twenty-five counterparties.

In [4]:
print(f"trades          : {len(df):,}")
print(f"window          : {df.trade_date.min():%Y-%m-%d} to {df.trade_date.max():%Y-%m-%d}")
print(f"gross notional  : ${df.notional_value.sum() / 1e9:,.2f}bn")
print(f"traders / desks : {df.trader.nunique()} / {df.desk.nunique()}")
print(f"counterparties  : {df.counterparty.nunique()}")
print(f"amended         : {int(df.amendment_flag.sum()):,}")
print(f"cancelled       : {int(df.cancellation_flag.sum()):,}")

df.groupby(["desk", "product_type"]).agg(
    trades=("trade_id", "size"),
    notional_m=("notional_value", lambda s: s.sum() / 1e6),
    median_ticket_m=("notional_value", lambda s: s.median() / 1e6),
).round(1)

trades          : 3,600
window          : 2026-01-01 to 2026-06-30
gross notional  : $25.22bn
traders / desks : 15 / 5
counterparties  : 25
amended         : 464
cancelled       : 118


trades  notional_m  median_ticket_m
desk        product_type                                     
Commodities Option           138      179.10             0.50
            Swap             194    3,759.70             8.50
Credit      Bond             310    1,446.40             2.60
            Swap             152    2,165.80             7.90
Equities    Equity           521    1,391.60             1.10
            Option           198      246.00             0.60
FX          FX Forward       710    3,731.50             2.30
            Option           151      222.40             0.50
Rates       Bond             702    3,863.50             2.50
            Swap             524    8,217.90             7.60

### Notional distribution

Notional spans four orders of magnitude, so the distribution only reads properly on a log
axis. The median ticket is around \$2.4m while the largest is roughly \$750m — the kind
of spread that makes a *fixed* dollar threshold a blunt instrument and argues for the
percentile mode in the large-notional rule.

In [5]:
quantiles = df.notional_value.quantile([0.01, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0])
print((quantiles / 1e6).round(2).rename("notional ($m)").to_string())

import plotly.graph_objects as go
from src.config import COLORS, PLOTLY_TEMPLATE

fig = go.Figure()
fig.add_histogram(
    x=np.log10(df.notional_value),
    nbinsx=60,
    marker_color=COLORS["normal"],
    name="all trades",
)
for q in (0.5, 0.99):
    fig.add_vline(
        x=np.log10(df.notional_value.quantile(q)),
        line_dash="dash",
        line_color=COLORS["flagged"] if q == 0.99 else COLORS["neutral"],
        annotation_text=f"p{int(q * 100)}",
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Notional distribution (log10 scale)",
    xaxis_title="log10(notional)",
    yaxis_title="trades",
    height=380,
    showlegend=False,
)
fig.show()

0.01     0.06
0.25     1.02
0.50     2.37
0.75     5.54
0.90    12.91
0.95    21.56
0.99    71.74
1.00   749.96


In [6]:
trade_volume_trend_chart(df, metric="trade_count", freq="W", by_desk=True).show()

Activity drifts upward through the window and Rates is consistently the busiest desk —
worth holding on to, because it is exactly what the desk-level volume rule picks up in
section 3.3.

## 3. The five rules, one at a time

Each rule is a pure function over the cleaned frame, returning flags plus a summary table.
Below is the default behaviour of each on the full book.

### 3.1 Large notional

In [7]:
large = flag_large_notional(df, percentile=0.99, mode="percentile")
threshold = large.details["effective_threshold"]
share = large.details["flagged_notional"] / df.notional_value.sum() * 100

print(f"p99 threshold      : ${threshold / 1e6:,.1f}m")
print(f"trades flagged     : {large.n_flagged:,} ({large.n_flagged / len(df) * 100:.1f}%)")
print(f"share of notional  : {share:.1f}%")

large.summary.assign(
    threshold_m=lambda d: d.threshold / 1e6,
    max_notional_m=lambda d: d.max_notional / 1e6,
    flagged_notional_m=lambda d: d.flagged_notional / 1e6,
)[["product_type", "trades", "threshold_m", "max_notional_m", "flagged_trades", "flagged_notional_m"]].round(1)

p99 threshold      : $71.7m
trades flagged     : 36 (1.0%)
share of notional  : 30.6%


,product_type,trades,threshold_m,max_notional_m,flagged_trades,flagged_notional_m
0,Swap,870,71.70,750.00,20,"4,636.20"
1,Bond,1012,71.70,496.20,7,"1,418.70"
2,FX Forward,710,71.70,602.20,6,"1,365.90"
3,Equity,521,71.70,115.10,3,298.80
4,Option,487,71.70,65.20,0,0.00


The top 1% of tickets carry roughly 30% of gross notional. That is the injected outlier
tail doing its job, and it is also why this rule earns its place: three dozen tickets
account for a third of the firm's exposure.

### 3.2 Counterparty concentration

In [8]:
concentration = flag_counterparty_concentration(df, top_n=5, exposure_pct_threshold=7.5)
d = concentration.details

print(f"counterparties     : {d['n_counterparties']}")
print(f"largest exposure   : {d['top1_pct']:.1f}% of notional")
print(f"top 5 combined     : {d['top5_pct']:.1f}%")
print(f"HHI                : {d['hhi']:,.0f}")

concentration.summary.head(10).assign(notional_m=lambda x: x.notional / 1e6)[
    ["rank", "counterparty", "trades", "notional_m", "exposure_pct", "cumulative_pct", "flagged"]
].round(2)

counterparties     : 25
largest exposure   : 16.5% of notional
top 5 combined     : 44.9%
HHI                : 660


,rank,counterparty,trades,notional_m,exposure_pct,cumulative_pct,flagged
0,1,Chang-Peterson,512,"4,151.39",16.46,16.46,True
1,2,Fernandez-Ferguson,109,"1,943.61",7.71,24.16,True
2,3,Crawford-Garrett,259,"1,943.24",7.70,31.87,True
3,4,Larsen Ray and Young,322,"1,839.30",7.29,39.16,True
4,5,Powell Clark and Lopez,87,"1,447.33",5.74,44.90,True
5,6,Anthony-Baker,215,"1,353.84",5.37,50.26,False
6,7,Santiago Reyes and Lynch,93,"1,311.58",5.20,55.46,False
7,8,Foster Jones and Reed,207,"1,237.77",4.91,60.37,False
8,9,Stanton-Wilkinson,147,940.20,3.73,64.10,False
9,10,Bishop LLC,100,928.91,3.68,67.78,False


In [9]:
exposure_by_counterparty_chart(df, exposure=concentration.summary, top_n=15).show()

The largest counterparty faces about 16% of the book and the top five together just under
half, against an HHI of roughly 660. On the merger-guideline reading of HHI that is an
*unconcentrated* book — but a single name at 16% would still sit near or above a typical
single-name exposure limit, which is why the rule flags on share as well as rank rather
than relying on HHI alone.

### 3.3 Trader and desk volume

In [10]:
by_trader = flag_high_trade_volume(df, group_by="trader", z_threshold=2.0)
by_desk = flag_high_trade_volume(df, group_by="desk", z_threshold=1.2)

print(f"traders flagged : {by_trader.details['n_flagged_groups']} of {by_trader.details['n_groups']}")
print(f"desks flagged   : {by_desk.details['n_flagged_groups']} of {by_desk.details['n_groups']}")
print(f"max attainable z with {by_desk.details['n_groups']} desks: {by_desk.details['max_attainable_z']:.2f}")

display(by_trader.summary[["group", "trade_count", "z_score", "flagged"]].head(6).round(2))
by_desk.summary[["group", "trade_count", "z_score", "flagged"]].round(2)

traders flagged : 1 of 15
desks flagged   : 1 of 5
max attainable z with 5 desks: 2.00


,group,trade_count,z_score,flagged
0,Tracy Hayes,590,2.27,True
1,Sarah Silva,446,1.34,False
2,Pamela Hahn,401,1.04,False
3,Courtney Wagner,335,0.62,False
4,Raymond Williams,329,0.58,False
5,Kevin Gutierrez,285,0.29,False


,group,trade_count,z_score,flagged
0,Rates,1226,1.44,True
1,FX,861,0.40,False
2,Equities,719,-0.00,False
3,Credit,462,-0.73,False
4,Commodities,332,-1.11,False


Note the ceiling on the desk z-score. With a sample standard deviation the largest
attainable z across *n* groups is `sqrt(n - 1)`, which is exactly 2.00 for five desks — so
the 2.0 threshold that works fine for fifteen traders can *never* fire on desks, however
lopsided the book is. The dashboard therefore defaults desks to 1.2 and warns when the
threshold is set above the ceiling. Small-*n* peer groups need either a lower cut-off or a
robust method such as the IQR fence.

In [11]:
trader_desk_activity_chart(df, metric="trade_count").show()

### 3.4 Late amendments and cancellations

In [12]:
late = flag_late_amendment_cancellation(df, business_days_threshold=2)
d = late.details

print(f"lifecycle events : {d['n_lifecycle_events']:,} "
      f"({d['n_amendments']:,} amendments, {d['n_cancellations']:,} cancellations)")
print(f"median lag       : {d['median_lag_days']:.0f} business days")
print(f"booked late      : {late.n_flagged:,} ({d['late_event_rate_pct']:.1f}% of events)")

late.summary.head(10).round(1)

lifecycle events : 464 (346 amendments, 118 cancellations)
median lag       : 1 business days
booked late      : 55 (11.9% of events)


,desk,event_type,events,late_events,median_lag_days,max_lag_days,late_rate_pct
0,Rates,Amendment,113,20,1.00,15.00,17.70
1,Commodities,Amendment,34,9,1.00,15.00,26.50
2,Equities,Amendment,71,7,1.00,12.00,9.90
3,FX,Amendment,89,7,0.00,14.00,7.90
4,Credit,Amendment,39,4,1.00,15.00,10.30
5,FX,Cancellation,26,4,1.00,12.00,15.40
6,Rates,Cancellation,52,3,1.00,13.00,5.80
7,Equities,Cancellation,21,1,0.00,11.00,4.80
8,Commodities,Cancellation,8,0,0.50,1.00,0.00
9,Credit,Cancellation,11,0,0.00,1.00,0.00


In [13]:
lags = df.loc[df.amendment_flag, "days_to_amendment"]
lags.value_counts().sort_index().rename("events").to_frame().T

days_to_amendment,0.00,1.00,2.00,3.00,4.00,5.00,6.00,7.00,8.00,9.00,10.00,11.00,12.00,13.00,14.00,15.00
events,215,134,60,5,6,4,2,4,3,4,3,3,6,6,6,3


Most amendments land same-day or next-day, as they should. The tail beyond T+2 is the
interesting part: ~12% of events, stretching out to fifteen business days. Rates
contributes the largest share of them, which lines up with it being the busiest desk.

### 3.5 Settlement risk

In [14]:
settlement = flag_settlement_risk(df)
d = settlement.details

print(f"too fast              : {d['n_too_fast']:,}")
print(f"too slow              : {d['n_too_slow']:,}")
print(f"dated before the trade: {d['n_before_trade_date']:,}")
print("\nexpected windows (business days):")
for product, (lo, hi) in PRODUCT_SETTLEMENT_WINDOW.items():
    print(f"  {product:<12} T+{lo}..T+{hi}")

settlement.summary

too fast              : 21
too slow              : 21
dated before the trade: 9

expected windows (business days):
  Bond         T+1..T+2
  Swap         T+1..T+3
  Equity       T+1..T+2
  Option       T+1..T+2
  FX Forward   T+15..T+70


,product_type,trades,expected_min,expected_max,median_days,too_fast,too_slow,missing_date,flagged_trades
0,Bond,1012,1.00,2.00,1.00,6,10,0,16
1,Option,487,1.00,2.00,1.00,5,5,0,10
2,FX Forward,710,15.00,70.00,42.00,5,3,0,8
3,Swap,870,1.00,3.00,2.00,3,2,0,5
4,Equity,521,1.00,2.00,1.00,2,1,0,3


In [15]:
settlement_timing_chart(run_all_rules(df)).show()

Applying a single flat window here would be wrong. FX Forwards settle one to three months
out by construction, so a flat T+1..T+3 test would flag every one of the 710 forwards as a
breach while missing nothing real. Per-product windows reduce that to a handful of genuine
outliers — including a few tickets dated to settle *before* they were traded, which is
only ever a keying error.

## 4. All five rules together

In [16]:
flagged = run_all_rules(df, SurveillanceConfig())

counts = pd.DataFrame(
    {
        "rule": [FLAG_LABELS[c] for c in FLAG_COLUMNS],
        "trades_flagged": [int(flagged[c].sum()) for c in FLAG_COLUMNS],
    }
)
counts["pct_of_book"] = counts.trades_flagged / len(flagged) * 100
counts.round(1)

,rule,trades_flagged,pct_of_book
0,Large notional,36,1.00
1,Counterparty concentration,1289,35.80
2,High volume (trader),590,16.40
3,High volume (desk),1226,34.10
4,Late amendment / cancellation,55,1.50
5,Settlement risk,42,1.20


In [17]:
trade_level = flagged[list(TRADE_LEVEL_FLAGS)].any(axis=1)
entity_level = flagged[list(ENTITY_LEVEL_FLAGS)].any(axis=1)

print(f"trade-level exception rate : {trade_level.mean() * 100:.1f}%")
print(f"entity-level alert rate    : {entity_level.mean() * 100:.1f}%")
print(f"any exception              : {(flagged.exception_count > 0).mean() * 100:.1f}%")
print()
print(flagged.exception_count.value_counts().sort_index().rename("trades").to_string())

trade-level exception rate : 3.6%
entity-level alert rate    : 57.8%
any exception              : 59.1%

exception_count
0    1471
1    1273
2     616
3     227
4      13


This split matters more than any single number on the page. The three **trade-level** rules
— large notional, late amendment, settlement breach — are properties of an individual
ticket and between them flag 3.6% of the book, which is a workable review queue. The three
**entity-level** rules flag a counterparty, a trader or a desk, and then necessarily tag
*every* trade that entity touched: 57.8% of the book, driven almost entirely by the Rates
desk and the top-five counterparties.

Reporting the combined figure as one "exception rate" would be actively misleading, so the
dashboard reports the two separately and counts entity-level findings as flagged
*entities* rather than flagged trades.

In [18]:
exception_summary(flagged, group_by="desk")[
    ["desk", "trades", "flagged_trades", "total_exceptions", "exception_rate_pct"]
].round(1)

,desk,trades,flagged_trades,total_exceptions,exception_rate_pct
0,Rates,1226,1226,2305,100.00
1,FX,861,354,365,41.10
2,Equities,719,272,279,37.80
3,Credit,462,144,152,31.20
4,Commodities,332,133,137,40.10


In [19]:
exception_summary(flagged, group_by="product_type")[
    ["product_type", "trades", "exception_rate_pct", "flag_large_notional",
     "flag_settlement_risk", "flag_late_amendment"]
].round(1)

,product_type,trades,exception_rate_pct,flag_large_notional,flag_settlement_risk,flag_late_amendment
0,Bond,1012,79.00,7,16,22
1,Swap,870,75.10,20,5,9
2,FX Forward,710,41.40,6,8,10
3,Equity,521,37.00,3,3,6
4,Option,487,39.00,0,10,8


Rates shows a 100% exception rate purely because the desk itself is the volume outlier —
a clean demonstration of the fan-out effect above, not a sign that every Rates ticket is
problematic.

## 5. The same aggregates, in SQL

The cleaned frame is loaded into an in-memory SQLite table so the headline aggregates can
be expressed as the saved views they would be in a real surveillance warehouse. The
dashboard shows these queries to the analyst alongside their results.

In [20]:
display(desk_exception_league_table(flagged))
monthly_exception_trend(flagged)

,desk,trades,notional_millions,flagged_trades,total_exceptions,exception_rate_pct,large_notional,late_amendments,settlement_breaches
0,Rates,1226,"12,081.40",1226,2305,100.00,16,23,16
1,FX,861,"3,953.90",354,365,41.10,6,11,11
2,Equities,719,"1,637.60",272,279,37.80,3,8,7
3,Credit,462,"3,612.30",144,152,31.20,7,4,4
4,Commodities,332,"3,938.90",133,137,40.10,4,9,4


,trade_month,trades,flagged_trades,exception_rate_pct,notional_millions
0,2026-01,522,319,61.10,"2,670.10"
1,2026-02,512,291,56.80,"4,090.90"
2,2026-03,591,332,56.20,"4,383.60"
3,2026-04,620,379,61.10,"4,334.30"
4,2026-05,643,390,60.70,"5,124.40"
5,2026-06,712,418,58.70,"4,620.80"


## 6. Findings

1. **Exposure is top-heavy but not extreme.** The largest counterparty faces ~16% of gross
   notional and the top five just under half, at an HHI of ~660. The book is diversified in
   aggregate, but a single name at 16% is the sort of position a credit committee would
   want sized deliberately rather than by accident.
2. **A third of the firm's notional sits in 1% of the tickets.** Thirty-six trades above
   the ~\$72m p99 threshold carry ~30% of gross notional. Any control keyed to trade
   *count* rather than size will systematically under-weight where the risk actually is.
3. **The genuinely actionable queue is small.** Trade-level exceptions run at 3.6% of the
   book — about 130 tickets over six months, or roughly one a day. That is a review load an
   ops team can absorb, which is the test any surveillance rule set has to pass.
4. **Operational risk clusters where the volume is.** Rates is both the busiest desk and
   the largest single contributor of late amendments, and Bonds and Swaps carry most of the
   settlement breaches. Volume and error rate move together, which argues for
   volume-adjusted rather than absolute desk-level control targets.
5. **Thresholds have to be product-aware.** A flat T+1..T+3 settlement window would flag
   all 710 FX Forwards and tell an analyst nothing; per-product windows cut that to ~40
   real outliers. The same logic applies to notional, where a fixed dollar threshold cannot
   serve a book whose median ticket varies by two orders of magnitude across products.

### Where this would go next

The rules here are univariate and unsupervised, which is the right starting point but not
the finish line. The obvious extensions are a *combined* score that weights the six flags
by historical false-positive rate instead of counting them equally, peer-group comparison
over time rather than a single cross-section (a trader who doubles their own run-rate is
more interesting than one who is simply busy), and a feedback loop that records which
alerts an analyst dismissed so thresholds can be tuned against outcomes rather than
intuition.